# Bambara SLM v3: make synthetic data with the SFT v2 model (phase 3)

This notebook uses the SFT v2 model to make new training data:
- **Back-translation (bt):** about 60,000 real Bambara sentences are translated into French and English. The French is then translated back into Bambara, and we measure how close it is to the original (round-trip chrF).
- **Chat (chat):** human conversations (oasst2) and grounded questions (dolly) are translated into Bambara, message by message, with the same round-trip check.
- **Filter:** bad rows are removed (wrong language, bad length, lowest 40% round-trip score), and review sheets of 200 random rows per category are made.

**Before you start**
1. Settings (right panel): Accelerator **GPU T4 x2**, Internet **On**.
2. **+ Add Input** (right panel) and attach:
   - `bambara-slm-code`: upload a **new version** of `dist/bambara_slm_colab.zip` (it now contains `data/interim/v3/`, with `bt_sources.jsonl`).
   - `bambara-sft-v2`: the output of your SFT v2 notebook (it contains `results_sft_v2.zip`).
   - *Only to continue a stopped run:* the output of a previous run of **this** notebook (it contains `v3_generated.zip`). Work already done is skipped.
3. Click **Save Version → Save & Run All (Commit)**. You can close the browser; it runs alone.

**Time:** the notebook stops generating by itself after about 10 hours (Kaggle stops at 12 h) and always saves what it made. Cell 3 prints a first time estimate after a few minutes (look for `ESTIMATE`). If the work is not finished, run the notebook again with the previous output attached (step 2): it continues where it stopped.

**Result:** download `v3_generated.zip` from the **Output** tab and give it to Claude. It contains `v3_out/bt_out_filtered.jsonl`, `v3_out/chat_bam_filtered.jsonl`, `v3_out/v3_filter.stats.json`, the raw outputs, the logs, and the review sheets `v3_out/review/v3_*_sample.csv` (fill the column "decision (ok / fix / no)").

## 1. Setup: code, model, packages, previous outputs

In [ ]:
import glob, os, shutil, subprocess, sys, time, zipfile
T_START   = time.time()
MAX_HOURS = 4.0                  # stop generating after this; filter + zip take a few minutes more
STOP_AT   = T_START + MAX_HOURS * 3600
USE_VLLM  = True                  # try vLLM first (much faster); falls back to HF transformers if it fails
WORK      = "/kaggle/working"
OUT       = f"{WORK}/v3_out"      # all outputs (append-only, resumable)
os.makedirs(OUT, exist_ok=True)

def find_input(keyword, required=True):
    hits = [p for p in glob.glob("/kaggle/input/**/", recursive=True) if keyword in p.lower()]
    if not hits:
        assert not required, f"no input matching '{keyword}': click 'Add Input' and attach it"
        return None
    return sorted(hits, key=len)[0].rstrip("/")

def unpack(src, dst):
    """Kaggle input (zip or already-extracted folder) -> writable folder."""
    shutil.rmtree(dst, ignore_errors=True)
    zips = glob.glob(f"{src}/**/*.zip", recursive=True)
    if zips:
        for z in zips:
            zipfile.ZipFile(z).extractall(dst)
    else:
        shutil.copytree(src, dst)
    return dst

# 1. code + inputs (data/interim/v3)
CODE = unpack(find_input("bambara-slm-code"), "/tmp/code")
CODE = os.path.dirname(sorted(glob.glob(f"{CODE}/**/pyproject.toml", recursive=True), key=len)[0])
IN = f"{CODE}/data/interim/v3"
assert os.path.exists(f"{CODE}/src/bslm/v3data/generate.py"), "old bundle: upload the new bambara_slm_colab.zip"
assert os.path.exists(f"{IN}/bt_sources.jsonl"), "bundle without data/interim/v3/bt_sources.jsonl: rebuild it"
print("code:", CODE, "| inputs:", sorted(os.listdir(IN)))

# 2. model = merged SFT v2 (results_sft_v2.zip inside the 'bambara-sft-v2' notebook output)
sft_in = find_input("bambara-sft-v2")
cfgs = glob.glob(f"{sft_in}/**/merged/config.json", recursive=True)
if not cfgs:                                         # still zipped
    for z in glob.glob(f"{sft_in}/**/results_sft_v2*.zip", recursive=True) or glob.glob(f"{sft_in}/**/*.zip", recursive=True):
        with zipfile.ZipFile(z) as zf:
            members = [m for m in zf.namelist() if "/merged/" in m]
            zf.extractall("/tmp/sft_v2", members=members)
    cfgs = glob.glob("/tmp/sft_v2/**/merged/config.json", recursive=True)
assert cfgs, "no merged/ folder found in the bambara-sft-v2 input"
MODEL = os.path.dirname(cfgs[0])
print("model:", MODEL, sorted(os.listdir(MODEL)))

# 3. packages (the system Python is used by the HF backend and by the filter)
subprocess.run('pip install -q "transformers==5.22.1" datasketch sacrebleu pyyaml "huggingface_hub>=0.30"', shell=True)
# fastText (language ID): prebuilt wheels first; the source builds fail on recent Kaggle images
for pkg in ('fasttext-numpy2-wheel', 'fasttext-wheel', 'fasttext'):
    if subprocess.run(f'pip install -q {pkg}', shell=True).returncode == 0 and \
       subprocess.run('python -c "import fasttext"', shell=True).returncode == 0:
        print('fasttext installed via', pkg); break
else:
    raise SystemExit('fasttext could not be installed')
subprocess.run("pip uninstall -y -q torchao", shell=True)

# 4. continue a previous run: copy its raw outputs (ids already done are skipped)
prev = sorted(glob.glob("/kaggle/input/**/v3_generated.zip", recursive=True))
if not prev:   # Kaggle sometimes shows a previous output already extracted
    prev = sorted({os.path.dirname(f) for f in glob.glob("/kaggle/input/**/v3_out/bt_out*.jsonl", recursive=True)
                   + glob.glob("/kaggle/input/**/v3_out/chat_bam*.jsonl", recursive=True)})
if prev:
    tmp = "/tmp/prev_v3"
    shutil.rmtree(tmp, ignore_errors=True)
    for k, p in enumerate(prev):
        if p.endswith(".zip"):
            zipfile.ZipFile(p).extractall(f"{tmp}/{k}")
        else:
            shutil.copytree(p, f"{tmp}/{k}")
    n = 0
    for f in glob.glob(f"{tmp}/**/*.jsonl", recursive=True):
        b = os.path.basename(f)
        if (b.startswith("bt_out") or b.startswith("chat_bam")) and "_filtered" not in b:
            shutil.copy(f, f"{OUT}/{b}"); n += 1
    print(f"resuming: copied {n} output files from {prev}:", sorted(os.listdir(OUT)))
else:
    print("fresh run (no previous output attached)")

import torch
NGPU = max(1, torch.cuda.device_count())
assert torch.cuda.is_available(), "No GPU: Settings -> Accelerator -> GPU T4 x2, then restart"
print("GPUs:", NGPU, [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
ENV = {**os.environ, "PYTHONPATH": f"{CODE}/src", "PYTHONUNBUFFERED": "1", "TOKENIZERS_PARALLELISM": "false"}
def gen_cmd(py, *args):
    return [py, "-m", "bslm.v3data.generate", *args, "--model", MODEL, "--in-dir", IN, "--out-dir", OUT]

## 2. Choose the backend: vLLM if it installs and works on a T4 (fp16), else HF transformers

In [ ]:
BACKEND, PY = "hf", sys.executable
if USE_VLLM:
    venv = "/tmp/venv-vllm"
    try:
        subprocess.run("pip install -q uv", shell=True, check=True)
        subprocess.run(f"uv venv -q --python {sys.version_info.major}.{sys.version_info.minor} {venv}", shell=True, check=True)
        subprocess.run(f"uv pip install -q --python {venv}/bin/python vllm datasketch sacrebleu pyyaml",
                       shell=True, check=True, timeout=1500)
        subprocess.run(f"uv pip install -q --python {venv}/bin/python fasttext-wheel", shell=True)  # optional: else the filter scores langid
        r = subprocess.run(gen_cmd(f"{venv}/bin/python", "selftest", "--backend", "vllm"), cwd=CODE, timeout=1200,
                           env={**ENV, "CUDA_VISIBLE_DEVICES": "0"}, capture_output=True, text=True)
        print("\n".join(l for l in (r.stdout + r.stderr).splitlines() if "selftest" in l or "Error" in l)[-3000:])
        if r.returncode == 0:
            BACKEND, PY = "vllm", f"{venv}/bin/python"
    except Exception as e:
        print("vLLM not usable:", type(e).__name__, e)
if BACKEND == "hf":
    r = subprocess.run(gen_cmd(PY, "selftest", "--backend", "hf"), cwd=CODE, env={**ENV, "CUDA_VISIBLE_DEVICES": "0"},
                       capture_output=True, text=True)
    print(r.stdout[-3000:], r.stderr[-2000:])
    assert r.returncode == 0, "the HF selftest failed: see the error above"
print("BACKEND =", BACKEND)
subprocess.run([sys.executable, "-m", "bslm.v3data.generate", "plan", "--in-dir", IN, "--out-dir", OUT, "--gpus", str(NGPU),
                "--tok-s", "2500" if BACKEND == "vllm" else "700"], cwd=CODE, env=ENV)

## 3. Generate: chat then back-translation, one process per GPU (work split in shards)

Each GPU does its half of the chat data, then its half of the back-translation. Logs: `v3_out/logs/`. Progress is printed every 5 minutes; look for `ESTIMATE` (measured speed and time left).

In [ ]:
os.makedirs(f"{OUT}/logs", exist_ok=True)
procs = []
for g in range(NGPU):
    sh = f"{g}/{NGPU}"
    chat = " ".join(gen_cmd(PY, "chat", "--backend", BACKEND, "--shard", sh, "--stop-at", str(STOP_AT)))
    bt   = " ".join(gen_cmd(PY, "bt",   "--backend", BACKEND, "--shard", sh, "--stop-at", str(STOP_AT)))
    log = open(f"{OUT}/logs/gpu{g}.log", "a")
    procs.append(subprocess.Popen(f"cd '{CODE}' && {chat} ; {bt}", shell=True, executable="/bin/bash",
                                  env={**ENV, "CUDA_VISIBLE_DEVICES": str(g)}, stdout=log, stderr=subprocess.STDOUT))
    print(f"GPU {g}: started shard {sh}")

def tail(g, n=4):
    lines = open(f"{OUT}/logs/gpu{g}.log", errors="replace").read().splitlines()
    keep = [l for l in lines if any(k in l for k in ("ESTIMATE", "progress", "SAMPLE", "Error", "error", "stopping", "to do"))]
    return "\n".join(keep[-n:])

while any(p.poll() is None for p in procs):
    time.sleep(300)
    print(f"--- {(time.time() - T_START) / 3600:.2f} h")
    for g in range(NGPU):
        print(f"GPU {g}:", tail(g))
print("exit codes:", [p.returncode for p in procs], "(0 = done, 3 = stopped at the time limit: run again to continue)")
for g in range(NGPU):
    print(f"GPU {g} last lines:\n" + "\n".join(open(f"{OUT}/logs/gpu{g}.log", errors="replace").read().splitlines()[-6:]))

## 4. Filter and review sheets

In [ ]:
r = subprocess.run([sys.executable, "-m", "bslm.v3data.generate", "filter", "--out-dir", OUT,
                    "--review-dir", f"{OUT}/review"], cwd=CODE, env=ENV, capture_output=True, text=True)
print(r.stdout[-6000:], r.stderr[-3000:])

## 5. Pack the outputs → download `v3_generated.zip` from the Output tab

In [ ]:
with zipfile.ZipFile(f"{WORK}/v3_generated.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for f in sorted(glob.glob(f"{OUT}/**/*", recursive=True)):
        if os.path.isfile(f):
            z.write(f, os.path.relpath(f, WORK))
print(f"{WORK}/v3_generated.zip", round(os.path.getsize(f"{WORK}/v3_generated.zip") / 1e6, 1), "MB")
print("total time:", round((time.time() - T_START) / 3600, 2), "h")